In [1]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import numpy as np
import gradio as gr

c:\Users\eboyu\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Provide the "Blueprint" (The classes you used during training)
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1), nn.InstanceNorm2d(channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.InstanceNorm2d(channels)
        )
    def forward(self, x): return x + self.conv(x)

class TinyStyleNet(nn.Module):
    def __init__(self):
        super(TinyStyleNet, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 9, stride=1, padding=4), nn.InstanceNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.InstanceNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.InstanceNorm2d(128), nn.ReLU(inplace=True)
        )
        self.res_blocks = nn.Sequential(*[ResidualBlock(128) for _ in range(5)])
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1), nn.InstanceNorm2d(64), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.InstanceNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 3, 9, stride=1, padding=4)
        )
    def forward(self, x): return self.decoder(self.res_blocks(self.encoder(x)))

# 2. Setup Device and Load the "Brain" into the "Blueprint"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TinyStyleNet().to(device)
model.load_state_dict(torch.load("starry_night_model.pth", map_location=device))
model.eval() # Set to evaluation mode (turns off training features)

# 3. Define the Inference Function
def fast_style(input_image):
    # Notice we resize to 512 here for higher quality outputs!
    transform = transforms.Compose([transforms.Resize(512), transforms.ToTensor()])
    tensor = transform(input_image).unsqueeze(0).to(device)

    with torch.no_grad(): # Don't track gradients (makes it super fast and saves VRAM)
        output_tensor = model(tensor)

    # Convert back to an image
    output_tensor = output_tensor.cpu().squeeze().clamp(0, 1).numpy().transpose(1, 2, 0)
    return Image.fromarray((output_tensor * 255).astype(np.uint8))

# 4. Launch the UI
gr.Interface(
    fn=fast_style, 
    inputs=gr.Image(type="pil"), 
    outputs=gr.Image(type="pil"),
    title="Starry Night Generator",
    description="Upload any photo to instantly turn it into a Van Gogh masterpiece!"
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
